# Create DICOMweb App Shells

Mints the gateway and viewer app service principals **without** deploying
any code. Downstream `deploy_grants` then grants those SPs catalog / volume
/ Lakebase access before `deploy_gateway` / `deploy_viewer` push code —
otherwise the wheel install from the UC volume fails with USE CATALOG.

In [0]:
%pip install --upgrade databricks-sdk==0.88.0 -q
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [0]:
app_gateway_name = "pixels-dicomweb-gateway"
app_viewer_name = "pixels-dicomweb"

# Both apps need these scopes so the user token tunneled from viewer to gateway
# (X-Pixels-User-Token) carries SQL + files.files privileges for OBO calls.
USER_API_SCOPES = ["sql", "files.files"]

# Resolve App Source Paths

In [0]:
import os

_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicomweb_gateway_path = os.path.join(_repo_root, "apps", "dicom-web-gateway")
_dicomweb_path = os.path.join(_repo_root, "apps", "dicom-web")

print(f"Gateway source: {_dicomweb_gateway_path}")
print(f"Viewer source:  {_dicomweb_path}")

# Mint App Shells (idempotent)

In [0]:
from databricks.sdk.service.apps import (
    App,
    AppResource,
    AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
    ComputeSize,
)

sql_resource = AppResource(
    name="sql_warehouse",
    sql_warehouse=AppResourceSqlWarehouse(
        id=sql_warehouse_id,
        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
    ),
)

_existing_apps = {a.name for a in w.apps.list()}

def ensure_app(name, source_path, compute_size=None):
    if name in _existing_apps:
        print(f"App '{name}' exists \u2014 skipping create")
        return
    print(f"Creating app shell '{name}' \u2014 this may take a few minutes \u2026")
    kwargs = dict(
        default_source_code_path=source_path,
        resources=[sql_resource],
        user_api_scopes=USER_API_SCOPES,
    )
    if compute_size is not None:
        kwargs["compute_size"] = compute_size
    app = w.apps.create_and_wait(App(name, **kwargs))
    print(f"\u2713 App created: {app.url}")

ensure_app(app_gateway_name, _dicomweb_gateway_path, ComputeSize.LARGE)
ensure_app(app_viewer_name, _dicomweb_path)

In [0]:
print("Service principals:")
for name in (app_gateway_name, app_viewer_name):
    sp = w.apps.get(name).service_principal_client_id
    print(f"  {name}: {sp}")